# 02 · Outlier Detection and Treatment

Objective: detect extreme values in `adr`, `lead_time` and total length
of stay, assess whether they are plausible or errors, and decide what to
keep, correct or exclude, checking their effect on the cancellation rate.

Data: for now this notebook uses the original CSV (`hotels.csv`). Once the
cleaned CSV from `01_cleaning` is ready, just change `csv_path` and re-run
the notebook.

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
csv_path = "../data/hotels.csv"
df = pd.read_csv(csv_path)

In [3]:
df.shape

(119390, 32)

In [4]:
df.dtypes.to_frame("dtype")

,dtype
hotel,str
is_canceled,int64
lead_time,int64
arrival_date_year,int64
arrival_date_month,str
arrival_date_week_number,int64
arrival_date_day_of_month,int64
stays_in_weekend_nights,int64
stays_in_week_nights,int64
adults,int64


In [5]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
is_canceled,119390.0,0.370416,0.482918,0.00,0.00,0.000,1.0,1.0
lead_time,119390.0,104.011416,106.863097,0.00,18.00,69.000,160.0,737.0
arrival_date_year,119390.0,2016.156554,0.707476,2015.00,2016.00,2016.000,2017.0,2017.0
arrival_date_week_number,119390.0,27.165173,13.605138,1.00,16.00,28.000,38.0,53.0
arrival_date_day_of_month,119390.0,15.798241,8.780829,1.00,8.00,16.000,23.0,31.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.00,0.00,1.000,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.00,1.00,2.000,3.0,50.0
adults,119390.0,1.856403,0.579261,0.00,2.00,2.000,2.0,55.0
children,119386.0,0.103890,0.398561,0.00,0.00,0.000,0.0,10.0
babies,119390.0,0.007949,0.097436,0.00,0.00,0.000,0.0,10.0


## Variable selection

Not every numeric column can have outliers in a meaningful sense. A variable is analysed
with the IQR rule only if (1) it measures a quantity, so "much larger than the rest" makes
sense, and (2) it has enough distinct values and spread, so that Q3 − Q1 > 0.

| Variable | Analyse? | Reason |
|---|---|---|
| `lead_time` | **Yes** | Quantity (days), many distinct values, max (737) far above Q3 (160) |
| `adr` | **Yes** | Quantity (price), max 5,400 vs Q3 126, and a negative minimum |
| `stays_in_weekend_nights` | **Yes, combined** | Summed into `total_nights` (total length of stay) |
| `stays_in_week_nights` | **Yes, combined** | Summed into `total_nights` (total length of stay) |
| `is_canceled` | No | Binary target (0/1), not a quantity |
| `is_repeated_guest` | No | Binary (0/1) |
| `arrival_date_year` | No | Date component, not a quantity |
| `arrival_date_week_number` | No | Date component (week 53 is valid in ISO years such as 2015) |
| `arrival_date_day_of_month` | No | Date component |
| `agent` | No | Identifier code; agent 535 is not "more" than agent 9 |
| `company` | No | Identifier code |
| `adults` | No | Q1 = Q3 = 2 → IQR = 0; values such as 0 or 55 are a plausibility issue, passed to data cleaning |
| `children` | No | Q1 = Q3 = 0 → IQR = 0; values up to 10 passed to data cleaning |
| `babies` | No | Q1 = Q3 = 0 → IQR = 0; values up to 10 passed to data cleaning |
| `previous_cancellations` | No | Q1 = Q3 = 0 → IQR = 0; high values are possible (e.g. agencies) |
| `previous_bookings_not_canceled` | No | Q1 = Q3 = 0 → IQR = 0 |
| `booking_changes` | No | Q1 = Q3 = 0 → IQR = 0 |
| `days_in_waiting_list` | No | Q1 = Q3 = 0 → IQR = 0; long waits are rare but possible |
| `required_car_parking_spaces` | No | Q1 = Q3 = 0 → IQR = 0 |
| `total_of_special_requests` | No | Small bounded range (0–5); no extreme values |

**Variables analysed:** `adr`, `lead_time` and `total_nights`
(= `stays_in_weekend_nights` + `stays_in_week_nights`).

In [6]:
# I will need only these columns for the outlier detection and treatment
required_columns = [
    "hotel",
    "is_canceled",
    "market_segment",
    "adr",
    "lead_time",
    "stays_in_weekend_nights",
    "stays_in_week_nights",
]

In [7]:
missing = []
for column in required_columns:
    if column not in df.columns:
        missing.append(column)
        
if len(missing) > 0:
    raise ValueError(f"Missing columns: {missing}")

In [8]:
#Check if the numerical variables are entirely numerical
numerical_columns = [
    'adr',
    'lead_time',
    'stays_in_weekend_nights',
    'stays_in_week_nights',
]

In [9]:
numerical_error = []
for column in numerical_columns:
    if not pd.api.types.is_numeric_dtype(df[column]):
        numerical_error.append(column)
        
if len(numerical_error) > 0:
    raise ValueError(f"Wrong numerical columns: {numerical_error}")

In [10]:
df['total_nights'] = df['stays_in_weekend_nights'] + df['stays_in_week_nights']

In [11]:
df[['total_nights', 'stays_in_weekend_nights', 'stays_in_week_nights']] 

,total_nights,stays_in_weekend_nights,stays_in_week_nights
0,0,0,0
1,0,0,0
2,1,0,1
3,1,0,1
4,2,0,2
...,...,...,...
119385,7,2,5
119386,7,2,5
119387,7,2,5
119388,7,2,5


In [12]:
df[['total_nights', 'stays_in_weekend_nights', 'stays_in_week_nights']].describe().T

,count,mean,std,min,25%,50%,75%,max
total_nights,119390.0,3.427900,2.557439,0.0,2.0,3.0,4.0,69.0
stays_in_weekend_nights,119390.0,0.927599,0.998613,0.0,0.0,1.0,2.0,19.0
stays_in_week_nights,119390.0,2.500302,1.908286,0.0,1.0,2.0,3.0,50.0


In [13]:
df[['adr', 'lead_time', 'total_nights']].isna().sum()

adr             0
lead_time       0
total_nights    0
dtype: int64

In [14]:
df[['adr', 'lead_time', 'total_nights']].count()

adr             119390
lead_time       119390
total_nights    119390
dtype: int64

With the original CSV there are no missing values in `adr`, `lead_time` or `total_nights`,
so all 119,390 rows are used to compute quantiles. This must be re-checked with the cleaned CSV.